# Multilingual Intent Classifier — Translate → Merge → fastText

Three stages in one notebook:

1. **Translate** the merged English dataset into **Arabic, French and Urdu** with NLLB-200.
2. **Merge** all four languages into a single dataset with a `language` column.
3. **Train** a single fastText classifier that handles all four languages at once.

### Two things to decide before running

**Translation cost.** 17,473 English rows × 3 languages ≈ 52k sentences. On a T4 with the distilled
600M NLLB model and greedy decoding that is roughly 45–90 minutes. Every stage checkpoints to disk and
resumes, so an interrupted run picks up where it stopped. Set `SMOKE_TEST = True` in the config cell
to do a 20-rows-per-intent trial pass first — strongly recommended before committing to the full run.

**Machine translation is not native phrasing.** Translated queries carry English word order and
idiom; "wake my card up" will not come back as something an Urdu speaker would actually type. The
classifier learns the translationese, so real-world accuracy on organic Arabic/Urdu input will be
lower than the test numbers here suggest. Treat this dataset as a strong starting point and plan to
mix in real user queries per language later. Section 6 has quality checks to catch the worst of it.

### What is done about overfitting and leakage

| Guard | What it does |
|---|---|
| **Split by source query, not by row** | All four translations of one query land in the *same* split. Without this, English "block card" in train and its Urdu twin in test is straight leakage |
| Stratified on intent | Every intent keeps its proportion in train / valid / test |
| De-duplication before and after normalisation | Translation collapses distinct English rows into identical target strings — those get removed |
| `autotune` scores against validation only | Hyperparameters can't be tuned into memorising |
| Train-vs-validation gap + epoch sweep | Numeric and visual read on memorisation |
| Per-language accuracy breakdown | Catches a model that is strong in English and coasting elsewhere |

## 1. Setup and configuration

In [ ]:
# !pip -q install transformers sentencepiece torch pandas scikit-learn matplotlib seaborn
# !pip -q install fasttext || pip -q install fasttext-wheel

import os
import re
import json
import time
import random
import unicodedata

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---------------------------------------------------------------- config
DATA_PATH = "merged_intent_dataset.csv"      # English source: columns query, intent
WORK_DIR = "./multilingual_intent"
TRANS_DIR = f"{WORK_DIR}/translations"
os.makedirs(TRANS_DIR, exist_ok=True)

SMOKE_TEST = False        # True -> only SMOKE_N rows per intent, for a fast end-to-end trial
SMOKE_N = 20

TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"   # 1.3B version is better but ~3x slower
BATCH_SIZE = 64
NUM_BEAMS = 1             # 1 = greedy (fast). 4 = better quality, roughly 3x the time.
MAX_NEW_TOKENS = 96

# NLLB language codes
LANGS = {
    "en": "eng_Latn",
    "ar": "arb_Arab",
    "fr": "fra_Latn",
    "ur": "urd_Arab",
}
TARGET_LANGS = ["ar", "fr", "ur"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))
print("smoke test:", SMOKE_TEST)

## 2. Load and clean the English source

Same cleaning as before, plus a `source_id` on every row. That id is what keeps a query and its three
translations together when the data is split later.

In [ ]:
df_en = pd.read_csv(DATA_PATH)
print("raw rows:", len(df_en))

df_en["query"] = df_en["query"].astype(str).str.strip()
df_en["intent"] = df_en["intent"].astype(str).str.strip()
df_en = df_en[(df_en["query"] != "") & (df_en["intent"] != "") &
              (df_en["intent"].str.lower() != "nan")]

before = len(df_en)
df_en = df_en.drop_duplicates(subset=["query", "intent"])
print("exact duplicate pairs dropped:", before - len(df_en))

conflicts = df_en.groupby(df_en["query"].str.lower())["intent"].nunique()
conflicting = set(conflicts[conflicts > 1].index)
if conflicting:
    print("conflicting queries dropped:", len(conflicting))
    df_en = df_en[~df_en["query"].str.lower().isin(conflicting)]

df_en = df_en.reset_index(drop=True)

if SMOKE_TEST:
    df_en = (df_en.sample(frac=1.0, random_state=SEED)
                  .groupby("intent", group_keys=False)
                  .head(SMOKE_N)
                  .reset_index(drop=True))
    print(f"SMOKE TEST -> {len(df_en)} rows")

df_en["source_id"] = np.arange(len(df_en))
df_en["language"] = "en"
df_en = df_en[["source_id", "query", "intent", "language"]]

LABELS = sorted(df_en["intent"].unique())
print("clean rows:", len(df_en), "| intents:", len(LABELS))
df_en["intent"].value_counts()

## 3. Translation model

NLLB-200 covers all three targets from one checkpoint, which matters for Urdu — the Helsinki
`opus-mt-en-ur` pair is noticeably weaker. Section 5 has commented alternatives if you would rather
use a translation API.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t0 = time.time()
tok = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
mt_model.eval()
print(f"loaded in {time.time() - t0:.1f}s")


def target_bos_id(tokenizer, lang_code):
    """Forced BOS token for the target language - the attribute moved between versions."""
    if hasattr(tokenizer, "lang_code_to_id"):
        return tokenizer.lang_code_to_id[lang_code]
    return tokenizer.convert_tokens_to_ids(lang_code)


@torch.no_grad()
def translate_batch(texts, src_lang, tgt_lang):
    tok.src_lang = src_lang
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    out = mt_model.generate(
        **enc,
        forced_bos_token_id=target_bos_id(tok, tgt_lang),
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
    )
    return tok.batch_decode(out, skip_special_tokens=True)


# sanity check on a handful of queries before committing to the full run
sample = df_en["query"].head(5).tolist()
for lang in TARGET_LANGS:
    print(f"\n--- {lang} ---")
    for src, tgt in zip(sample, translate_batch(sample, LANGS["en"], LANGS[lang])):
        print(f"  {src}\n  -> {tgt}")

## 4. Translate — with checkpointing

Only **unique** query strings are translated, then mapped back onto the rows. Progress is flushed to
`translations/<lang>.csv` every `CHECKPOINT_EVERY` batches, and re-running this cell skips anything
already done. Safe to interrupt.

In [ ]:
CHECKPOINT_EVERY = 20   # batches between disk flushes


def translate_column(texts, tgt_code, cache_path):
    """Translate a list of unique strings, resuming from cache_path if it exists."""
    cache = {}
    if os.path.exists(cache_path):
        cached_df = pd.read_csv(cache_path)
        cache = dict(zip(cached_df["source"].astype(str), cached_df["translation"].astype(str)))
        print(f"  resuming: {len(cache)} already translated")

    todo = [t for t in texts if t not in cache]
    print(f"  to translate: {len(todo)}")

    start = time.time()
    for bi, i in enumerate(range(0, len(todo), BATCH_SIZE)):
        batch = todo[i:i + BATCH_SIZE]
        try:
            translations = translate_batch(batch, LANGS["en"], tgt_code)
        except RuntimeError as e:                      # usually CUDA OOM - retry one by one
            print(f"  batch failed ({e}); falling back to single-item")
            translations = [translate_batch([b], LANGS["en"], tgt_code)[0] for b in batch]

        cache.update(dict(zip(batch, translations)))

        if bi % CHECKPOINT_EVERY == 0 or i + BATCH_SIZE >= len(todo):
            pd.DataFrame({"source": list(cache.keys()), "translation": list(cache.values())}) \
              .to_csv(cache_path, index=False)
            done = min(i + BATCH_SIZE, len(todo))
            rate = done / max(time.time() - start, 1e-6)
            eta = (len(todo) - done) / max(rate, 1e-6)
            print(f"  {done}/{len(todo)}  ({rate:.1f}/s, eta {eta / 60:.1f} min)", flush=True)

    return cache


unique_queries = df_en["query"].unique().tolist()
print(f"{len(unique_queries)} unique English queries\n")

translated_frames = {"en": df_en.copy()}

for lang in TARGET_LANGS:
    print(f"=== {lang} ({LANGS[lang]}) ===")
    cache_path = f"{TRANS_DIR}/{lang}.csv"
    mapping = translate_column(unique_queries, LANGS[lang], cache_path)

    frame = df_en.copy()
    frame["query"] = frame["query"].map(mapping)
    frame["language"] = lang
    translated_frames[lang] = frame
    print(f"  done: {frame['query'].notna().sum()} rows\n")

## 5. Alternative translation backends

Only if NLLB is not an option. Google via `deep-translator` is rate-limited and will take hours on 52k
sentences; Marian is fast but its Urdu pair is weak. Left commented out.

In [ ]:
# --- Option A: Google Translate via deep-translator (no GPU, rate-limited) ---
# !pip -q install deep-translator
# from deep_translator import GoogleTranslator
# from concurrent.futures import ThreadPoolExecutor
#
# def google_translate_all(texts, target):
#     tr = GoogleTranslator(source="en", target=target)
#     with ThreadPoolExecutor(max_workers=8) as pool:      # keep this low or you get blocked
#         return list(pool.map(tr.translate, texts))
#
# # google codes: ar, fr, ur

# --- Option B: Helsinki MarianMT, one model per pair (fast, weaker Urdu) ---
# MARIAN = {"ar": "Helsinki-NLP/opus-mt-en-ar",
#           "fr": "Helsinki-NLP/opus-mt-en-fr",
#           "ur": "Helsinki-NLP/opus-mt-en-ur"}

## 6. Translation quality checks

Four things worth looking at before training on this data:

* **empty or failed** rows,
* **unchanged** rows — output identical to the English input, meaning the model passed the string
  through (common for `qr`, `amc`, brand names; a high rate signals a real problem),
* **script coverage** — Arabic/Urdu rows should be in Arabic script, French in Latin,
* **length ratio** — a translation many times longer than its source is usually a repetition loop.

In [ ]:
AR_SCRIPT = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
LATIN = re.compile(r"[A-Za-z]")


def script_ratio(text, pattern):
    text = str(text)
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(bool(pattern.match(c)) for c in letters) / len(letters)


report = []
for lang, frame in translated_frames.items():
    if lang == "en":
        continue
    src = df_en["query"].tolist()
    tgt = frame["query"].astype(str).tolist()

    expected = AR_SCRIPT if lang in ("ar", "ur") else LATIN
    ratios = [script_ratio(t, expected) for t in tgt]
    len_ratio = [len(t) / max(len(s), 1) for s, t in zip(src, tgt)]

    report.append({
        "lang": lang,
        "rows": len(tgt),
        "empty": sum(1 for t in tgt if not t.strip() or t.lower() == "nan"),
        "unchanged_%": round(100 * np.mean([s.strip().lower() == t.strip().lower()
                                            for s, t in zip(src, tgt)]), 2),
        "in_expected_script_%": round(100 * np.mean([r > 0.5 for r in ratios]), 2),
        "median_len_ratio": round(float(np.median(len_ratio)), 2),
        "len_ratio_gt_3_%": round(100 * np.mean([r > 3 for r in len_ratio]), 2),
    })

print(pd.DataFrame(report).to_string(index=False))

In [ ]:
# eyeball a few translations per intent - the fastest way to catch systematic mistranslation
inspect = (df_en.sample(frac=1.0, random_state=SEED)
                .groupby("intent", group_keys=False)
                .head(1)
                .reset_index(drop=True))

for _, row in inspect.iterrows():
    print(f"[{row['intent']}]  {row['query']}")
    for lang in TARGET_LANGS:
        val = translated_frames[lang].loc[
            translated_frames[lang]["source_id"] == row["source_id"], "query"
        ].iloc[0]
        print(f"   {lang}: {val}")
    print()

## 7. Merge the four languages

Rows that came back empty, unchanged from English, or in the wrong script are dropped — a wrong-script
Urdu row is really just a duplicate English row wearing an Urdu label, and it teaches the model
nothing.

In [ ]:
frames = []
for lang, frame in translated_frames.items():
    f = frame.copy()
    f["query"] = f["query"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    f = f[(f["query"] != "") & (f["query"].str.lower() != "nan")]

    if lang != "en":
        expected = AR_SCRIPT if lang in ("ar", "ur") else LATIN
        keep = f["query"].apply(lambda t: script_ratio(t, expected) > 0.5)
        src_map = dict(zip(df_en["source_id"], df_en["query"].str.lower()))
        not_copy = f.apply(lambda r: r["query"].lower() != src_map.get(r["source_id"], ""), axis=1)
        dropped = (~(keep & not_copy)).sum()
        print(f"{lang}: dropping {dropped} rows (wrong script or unchanged from English)")
        f = f[keep & not_copy]

    frames.append(f[["source_id", "query", "intent", "language"]])

df_all = pd.concat(frames, ignore_index=True)

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["query", "intent"]).reset_index(drop=True)
print(f"\ncross-language duplicate rows dropped: {before - len(df_all)}")
print("merged rows:", len(df_all))

df_all.to_csv(f"{WORK_DIR}/multilingual_intent_dataset.csv", index=False)
print(pd.crosstab(df_all["intent"], df_all["language"], margins=True).to_string())

## 8. Unicode-safe normalisation

The English-only normaliser from the previous notebook stripped everything outside `[a-z0-9]`, which
would delete Arabic and Urdu entirely. This version works by Unicode **category** instead: keep
letters, marks and numbers in any script, drop punctuation and symbols.

Script-specific handling on top of that:

* NFKC normalisation, then removal of Arabic/Urdu diacritics (`\u064B–\u065F`), tatweel and
  zero-width joiners — these are inconsistently typed and only fragment the vocabulary.
* Arabic-Indic digits (`٤`, `۴`) mapped to ASCII so `5000` is one token everywhere.
* Optional unification of letter variants that differ between Arabic and Urdu conventions
  (`ي/ی`, `ك/ک`, `ه/ہ`, alef forms). This merges the two scripts' spellings of the same word, which
  shrinks the vocabulary and helps — the model only has to tell intents apart, not languages.

In [ ]:
UNIFY_ARABIC_URDU_VARIANTS = True

AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ZERO_WIDTH = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2066-\u2069\uFEFF]")
TATWEEL = "\u0640"

DIGIT_MAP = {}
for i in range(10):
    DIGIT_MAP[ord("\u0660") + i] = str(i)   # Arabic-Indic
    DIGIT_MAP[ord("\u06F0") + i] = str(i)   # Extended Arabic-Indic (Urdu/Persian)

VARIANT_MAP = str.maketrans({
    "\u0623": "\u0627", "\u0625": "\u0627", "\u0622": "\u0627",   # أ إ آ -> ا
    "\u064A": "\u06CC", "\u06D2": "\u06CC",                       # ي ے -> ی
    "\u0643": "\u06A9",                                           # ك -> ک
    "\u0647": "\u06C1", "\u06C3": "\u06C1",                       # ه ۃ -> ہ
    "\u0629": "\u06C1",                                           # ة -> ہ
})

KEEP_CHARS = set("'&")


def normalize(text: str) -> str:
    t = unicodedata.normalize("NFKC", str(text)).strip().lower()
    t = ZERO_WIDTH.sub("", t)
    t = AR_DIACRITICS.sub("", t)
    t = t.replace(TATWEEL, "")
    t = t.translate(DIGIT_MAP)
    if UNIFY_ARABIC_URDU_VARIANTS:
        t = t.translate(VARIANT_MAP)
    # keep Letters / Marks / Numbers in any script; everything else becomes a space
    t = "".join(c if (unicodedata.category(c)[0] in "LMN" or c in KEEP_CHARS) else " " for c in t)
    return re.sub(r"\s+", " ", t).strip()


df_all["text"] = df_all["query"].apply(normalize)
df_all = df_all[df_all["text"] != ""]

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text", "intent"]).reset_index(drop=True)
print("post-normalisation duplicates dropped:", before - len(df_all))
print("rows:", len(df_all))

for lang in ["en", "ar", "fr", "ur"]:
    row = df_all[df_all["language"] == lang].iloc[0]
    print(f"\n{lang}: {row['query']}\n -> {row['text']}")

## 9. Split by source query — not by row

The important step. Every language version of a query shares a `source_id`, so ids are split first and
rows follow. Split rows directly and the English original ends up training the model to answer its own
Urdu translation in the test set.

In [ ]:
from sklearn.model_selection import train_test_split

source_meta = df_all.groupby("source_id")["intent"].first().reset_index()

train_ids, temp_ids = train_test_split(
    source_meta, test_size=0.20, random_state=SEED, stratify=source_meta["intent"]
)
valid_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, random_state=SEED, stratify=temp_ids["intent"]
)

train_ids = set(train_ids["source_id"])
valid_ids = set(valid_ids["source_id"])
test_ids = set(test_ids["source_id"])

train_df = df_all[df_all["source_id"].isin(train_ids)].reset_index(drop=True)
valid_df = df_all[df_all["source_id"].isin(valid_ids)].reset_index(drop=True)
test_df = df_all[df_all["source_id"].isin(test_ids)].reset_index(drop=True)

print(f"source queries -> train {len(train_ids)}  valid {len(valid_ids)}  test {len(test_ids)}")
print(f"rows           -> train {len(train_df)}  valid {len(valid_df)}  test {len(test_df)}")

assert not (train_ids & valid_ids) and not (train_ids & test_ids), "source_id leaked across splits"
print("\ntext overlap train/test:", len(set(train_df['text']) & set(test_df['text'])))
print()
print(pd.crosstab(test_df["language"], test_df["intent"]).to_string())

## 10. Write the fastText files

`__label__<intent> <normalised text>`, one example per line. Languages are interleaved by shuffling so
no batch is single-language.

In [ ]:
TRAIN_FILE = f"{WORK_DIR}/train.txt"
VALID_FILE = f"{WORK_DIR}/valid.txt"
TEST_FILE = f"{WORK_DIR}/test.txt"


def write_fasttext_file(frame, path, shuffle=True):
    frame = frame.sample(frac=1.0, random_state=SEED) if shuffle else frame
    with open(path, "w", encoding="utf-8") as f:
        for text, intent in zip(frame["text"], frame["intent"]):
            f.write(f"__label__{intent} {text}\n")
    print(f"{path}: {len(frame)} lines")


write_fasttext_file(train_df, TRAIN_FILE)
write_fasttext_file(valid_df, VALID_FILE)
write_fasttext_file(test_df, TEST_FILE)

print()
print("".join(open(TRAIN_FILE, encoding="utf-8").readlines()[:5]))

## 11. Baseline fastText model

`minn`/`maxn` matter more here than in the English-only run: character n-grams are what let one model
share structure across four scripts and absorb the typos in the English rows.

Worth being explicit — fastText is trained from scratch, there is no pretrained checkpoint being
fine-tuned. The tuning that matters is the hyperparameter search in the next section.

In [ ]:
import fasttext

t0 = time.time()
baseline = fasttext.train_supervised(
    input=TRAIN_FILE,
    lr=0.5,
    epoch=25,
    wordNgrams=2,
    dim=100,
    minCount=1,
    minn=3,
    maxn=6,
    loss="softmax",
    bucket=400000,          # larger than the English-only run: 4 languages, bigger vocabulary
    thread=os.cpu_count(),
    seed=SEED,
    verbose=1,
)
print(f"trained in {time.time() - t0:.1f}s")
print("baseline validation accuracy:", round(baseline.test(VALID_FILE)[1], 4))

## 12. Hyperparameter search

In [ ]:
t0 = time.time()
model = fasttext.train_supervised(
    input=TRAIN_FILE,
    autotuneValidationFile=VALID_FILE,
    autotuneMetric="f1",
    autotuneDuration=600,       # 4 languages -> give it longer than the English-only run
    thread=os.cpu_count(),
    seed=SEED,
    verbose=2,
)
print(f"\nautotune finished in {time.time() - t0:.1f}s")

args = model.f.getArgs()
best = {
    "lr": args.lr, "dim": args.dim, "epoch": args.epoch, "wordNgrams": args.wordNgrams,
    "minCount": args.minCount, "minn": args.minn, "maxn": args.maxn,
    "bucket": args.bucket, "loss": str(args.loss),
}
for k, v in best.items():
    print(f"  {k:<12} {v}")

print(f"\ntuned      validation accuracy: {model.test(VALID_FILE)[1]:.4f}")
print(f"baseline   validation accuracy: {baseline.test(VALID_FILE)[1]:.4f}")

## 13. Overfitting checks

In [ ]:
train_acc = model.test(TRAIN_FILE)[1]
valid_acc = model.test(VALID_FILE)[1]
gap = train_acc - valid_acc

print(f"train accuracy     : {train_acc:.4f}")
print(f"validation accuracy: {valid_acc:.4f}")
print(f"gap                : {gap:.4f}  ->  "
      f"{'OK' if gap < 0.05 else 'overfitting - lower epoch/dim or raise minCount'}")

In [ ]:
import matplotlib.pyplot as plt

epochs_grid = [5, 10, 15, 25, 40, 60]
train_scores, valid_scores = [], []

for e in epochs_grid:
    m = fasttext.train_supervised(
        input=TRAIN_FILE, lr=best["lr"], dim=best["dim"], epoch=e,
        wordNgrams=best["wordNgrams"], minCount=best["minCount"],
        minn=best["minn"], maxn=best["maxn"], bucket=best["bucket"],
        thread=os.cpu_count(), seed=SEED, verbose=0,
    )
    train_scores.append(m.test(TRAIN_FILE)[1])
    valid_scores.append(m.test(VALID_FILE)[1])
    print(f"epoch={e:>3}  train={train_scores[-1]:.4f}  valid={valid_scores[-1]:.4f}  "
          f"gap={train_scores[-1] - valid_scores[-1]:.4f}")

plt.figure(figsize=(9, 5))
plt.plot(epochs_grid, train_scores, marker="s", label="train accuracy")
plt.plot(epochs_grid, valid_scores, marker="o", label="validation accuracy")
plt.xlabel("epochs")
plt.ylabel("accuracy")
plt.title("Where the curves separate is where memorisation starts")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 14. Test results — overall, then per language

The per-language table is the one to read. A high overall number hiding 97% English and 78% Urdu means
the Urdu translations are poor, not that the model is good.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def predict_frame(ft_model, frame):
    labels, probs = ft_model.predict(frame["text"].tolist(), k=1)
    return ([l[0].replace("__label__", "") for l in labels], [p[0] for p in probs])


y_true = test_df["intent"].tolist()
y_pred, y_conf = predict_frame(model, test_df)

print(f"test accuracy    : {accuracy_score(y_true, y_pred):.4f}")
print(f"test macro F1    : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"test weighted F1 : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

results_df = pd.DataFrame({
    "query": test_df["query"],
    "text": test_df["text"],
    "language": test_df["language"],
    "actual": y_true,
    "predicted": y_pred,
    "confidence": y_conf,
})
results_df["correct"] = results_df["actual"] == results_df["predicted"]

per_lang = results_df.groupby("language").agg(rows=("correct", "count"),
                                              correct=("correct", "sum"))
per_lang["accuracy_%"] = (100 * per_lang["correct"] / per_lang["rows"]).round(2)
per_lang["macro_f1"] = [
    round(f1_score(g["actual"], g["predicted"], average="macro", zero_division=0), 4)
    for _, g in results_df.groupby("language")
]
print()
print(per_lang.to_string())

In [ ]:
print(classification_report(y_true, y_pred, labels=LABELS, digits=4, zero_division=0))

In [ ]:
# intent x language accuracy heatmap - shows exactly which pairing is weak
pivot = (results_df.groupby(["actual", "language"])["correct"].mean() * 100).unstack()

import seaborn as sns

plt.figure(figsize=(8, 9))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", vmin=60, vmax=100,
            cbar_kws={"label": "accuracy %"})
plt.title("Accuracy by intent and language")
plt.xlabel("language")
plt.ylabel("intent")
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=LABELS)

plt.figure(figsize=(13, 11))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("predicted")
plt.ylabel("actual")
plt.title("Confusion matrix - test set, all languages")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 15. Errors

Grouping errors by language as well as by intent pair separates two very different problems: an intent
pair that is genuinely ambiguous (wrong in every language) versus one that only breaks in Urdu, which
points at the translation rather than the classifier.

In [ ]:
errors = results_df[~results_df["correct"]].sort_values("confidence", ascending=False)
print(f"{len(errors)} misclassified out of {len(results_df)}\n")

print("most common confusions:")
print(errors.groupby(["actual", "predicted"]).size().sort_values(ascending=False).head(15).to_string())

print("\nconfusions by language:")
print(pd.crosstab(errors["actual"] + " -> " + errors["predicted"], errors["language"])
      .assign(total=lambda d: d.sum(axis=1))
      .sort_values("total", ascending=False).head(12).to_string())

results_df.to_csv(f"{WORK_DIR}/test_predictions.csv", index=False)
errors[["query", "language", "actual", "predicted", "confidence"]].head(30)

## 16. Confidence threshold

Per language, because confidence is not calibrated the same way across them — the cut-off that works
for English may reject far too much Urdu.

In [ ]:
rows = []
for lang in ["all"] + sorted(results_df["language"].unique()):
    subset = results_df if lang == "all" else results_df[results_df["language"] == lang]
    for thr in [0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        kept = subset[subset["confidence"] >= thr]
        rows.append({
            "language": lang,
            "threshold": thr,
            "coverage_%": round(100 * len(kept) / len(subset), 1),
            "accuracy_on_kept_%": round(100 * kept["correct"].mean(), 2) if len(kept) else np.nan,
        })

threshold_df = pd.DataFrame(rows)
print(threshold_df.pivot(index="threshold", columns="language",
                         values="accuracy_on_kept_%").to_string())
print()
print(threshold_df.pivot(index="threshold", columns="language", values="coverage_%").to_string())

## 17. Quantise

In [ ]:
model.save_model(f"{WORK_DIR}/intent_model.bin")
full_size = os.path.getsize(f"{WORK_DIR}/intent_model.bin") / 1e6

quantized = fasttext.train_supervised(
    input=TRAIN_FILE, lr=best["lr"], dim=best["dim"], epoch=best["epoch"],
    wordNgrams=best["wordNgrams"], minCount=best["minCount"],
    minn=best["minn"], maxn=best["maxn"], bucket=best["bucket"],
    thread=os.cpu_count(), seed=SEED, verbose=0,
)
quantized.quantize(input=TRAIN_FILE, qnorm=True, retrain=True, cutoff=200000)
quantized.save_model(f"{WORK_DIR}/intent_model.ftz")
quant_size = os.path.getsize(f"{WORK_DIR}/intent_model.ftz") / 1e6

q_pred, _ = predict_frame(quantized, test_df)
print(f"full      : {full_size:8.2f} MB   test accuracy {accuracy_score(y_true, y_pred):.4f}")
print(f"quantised : {quant_size:8.2f} MB   test accuracy {accuracy_score(y_true, q_pred):.4f}")

## 18. Inference

The same normalisation runs at prediction time — skip it and Arabic/Urdu accuracy drops silently.
`detect_script` is a cheap heuristic for logging which language came in; the classifier itself does not
need it.

In [ ]:
URDU_SPECIFIC = set("\u0679\u0688\u0691\u06BA\u06D2\u06C1\u06AF\u0686\u067E\u0698\u06C2")
CONFIDENCE_THRESHOLD = 0.5


def detect_script(text):
    text = str(text)
    if any(c in URDU_SPECIFIC for c in text):
        return "ur"
    if AR_SCRIPT.search(text):
        return "ar"
    return "latin"          # English or French - fastText tells them apart, this heuristic doesn't


def predict(queries, k=1, threshold=CONFIDENCE_THRESHOLD, ft_model=None):
    """Single string or list. Returns (intent, confidence); intent is None below the threshold."""
    ft_model = ft_model or model
    single = isinstance(queries, str)
    raw = [queries] if single else list(queries)
    batch = [normalize(q) for q in raw]

    labels, probs = ft_model.predict(batch, k=k)

    out = []
    for lab, pr in zip(labels, probs):
        preds = [(l.replace("__label__", ""), round(float(p), 4)) for l, p in zip(lab, pr)]
        if k == 1:
            intent, conf = preds[0]
            out.append((intent if conf >= threshold else None, conf))
        else:
            out.append(preds)

    return out[0] if single else out


samples = [
    ("en", "block card"),
    ("en", "activete my new visa crad"),
    ("fr", "bloquer ma carte"),
    ("fr", "quel est le solde de mon compte"),
    ("ar", "\u0627\u0631\u064A\u062F \u062D\u0638\u0631 \u0628\u0637\u0627\u0642\u062A\u064A"),
    ("ar", "\u0645\u0627 \u0647\u0648 \u0631\u0635\u064A\u062F \u062D\u0633\u0627\u0628\u064A"),
    ("ur", "\u0645\u06CC\u0631\u0627 \u06A9\u0627\u0631\u0688 \u0628\u0644\u0627\u06A9 \u06A9\u0631\u06CC\u06BA"),
    ("ur", "\u0645\u06CC\u0631\u06D2 \u0627\u06A9\u0627\u0624\u0646\u0679 \u0645\u06CC\u06BA \u06A9\u062A\u0646\u06D2 \u067E\u06CC\u0633\u06D2 \u06C1\u06CC\u06BA"),
]

for lang, q in samples:
    intent, conf = predict(q)
    shown = intent if intent else "UNCERTAIN"
    print(f"[{lang}|{detect_script(q):<5}] {conf:.3f}  {shown:<24} {q}")

In [ ]:
# top-3 on an ambiguous one
for intent, conf in predict("payer ma facture d'electricite", k=3):
    print(f"{conf:.4f}  {intent}")

## 19. Save everything

In [ ]:
model.save_model(f"{WORK_DIR}/intent_model.bin")

config = {
    "labels": LABELS,
    "languages": ["en"] + TARGET_LANGS,
    "translation_model": TRANSLATION_MODEL,
    "best_params": {k: str(v) for k, v in best.items()},
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "unify_arabic_urdu_variants": UNIFY_ARABIC_URDU_VARIANTS,
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "per_language_accuracy": {k: float(v) for k, v in
                              (per_lang["accuracy_%"] / 100).to_dict().items()},
    "rows": {"train": len(train_df), "valid": len(valid_df), "test": len(test_df)},
}
with open(f"{WORK_DIR}/model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(json.dumps(config, indent=2, ensure_ascii=False)[:1200])
print()
print(sorted(os.listdir(WORK_DIR)))

In [ ]:
# Colab only
# import shutil
# from google.colab import files
# shutil.make_archive("multilingual_intent", "zip", WORK_DIR)
# files.download("multilingual_intent.zip")

## 20. Reload check

In [ ]:
reloaded = fasttext.load_model(f"{WORK_DIR}/intent_model.bin")
print(reloaded.test(TEST_FILE))
print(predict("bloquer ma carte bancaire", ft_model=reloaded))

---

### Reading the results

Expect a rough ordering of **English > French > Arabic ≈ Urdu**. English is the source, French is the
easiest target and shares a script, and the two Arabic-script languages sit lowest — partly translation
quality, partly a much larger effective vocabulary from rich morphology.

If Urdu or Arabic lags badly, work through in this order:

1. **Look at the translations, not the model.** The section 6 checks and the section 15 per-language
   confusion table will usually show it. Re-run translation with `NUM_BEAMS=4` or the 1.3B NLLB model
   for the worst intents.
2. **Widen the character n-grams** (`minn=2, maxn=7`). Arabic and Urdu morphology attaches a lot to the
   word stem, and subword coverage matters more than in English.
3. **Toggle `UNIFY_ARABIC_URDU_VARIANTS`** and re-run from section 8 — worth measuring both ways on
   your data rather than assuming.
4. **Raise `autotuneDuration`** to 900–1200s. Four languages is a bigger search space.
5. **Try per-language models** and compare against this single multilingual one. Four small models
   with a script-based router sometimes beat one shared model, at the cost of more to maintain.

The honest caveat stands: these numbers measure performance on *translated* text. Before shipping,
collect a few hundred real queries per language and evaluate against those — that gap is the one that
matters.